In [1]:
!nvidia-smi

Sat Sep  5 12:24:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [3]:
from google.colab import userdata
github_token = userdata.get("GITHUB_TOKEN")
!git clone https://{github_token}@github.com/D-L-C-S/contract-clause-qlora.git
%cd contract-clause-qlora
!pip install -q -e .
!pip install -q peft trl bitsandbytes

Cloning into 'contract-clause-qlora'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 37 (delta 13), reused 33 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 188.59 KiB | 5.71 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/contract-clause-qlora
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9

In [4]:
!git clone https://github.com/TheAtticusProject/cuad.git data/cuad-raw
!cd data/cuad-raw && unzip -o data.zip
!python scripts/build_dataset.py

Cloning into 'data/cuad-raw'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 30 (delta 10), reused 9 (delta 9), pack-reused 13 (from 1)
Receiving objects: 100% (30/30), 17.78 MiB | 32.80 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Archive:  data.zip
  inflating: CUADv1.json             
  inflating: test.json               
  inflating: train_separate_questions.json  
Loading CUAD...
  27633 examples (13823 positive, 13810 negative) across 510 contracts
Splitting by contract (no clause-level leakage)...
  train: 22017 rows / 408 contracts
  val:   2832 rows / 51 contracts
  test:  2784 rows / 51 contracts
Capping dominant categories in train at 80 (val/test left uncapped)...
  train: 22017 -> 2993 rows after capping
Loading tokenizer (microsoft/Phi-3-mini-4k-instruct)...
config.json: 100% 967/967 [00:00<00:00, 3.16MB/s]
configuration_phi3.py: 100% 11.2k/11.2k [00:00<00:00, 7.16MB

## Notes on training configuration

Two settings below aren't the obvious defaults — documenting why:

**`fp16=False`, `bf16=False` (full fp32 training, no mixed precision):**
Colab's free-tier T4 GPU (Turing architecture) has no native bf16 hardware
support. fp16 mixed precision was also ruled out — it triggered a real,
externally-documented bug (`NotImplementedError:
"_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for
'BFloat16'`, matching [pytorch#127176](https://github.com/pytorch/pytorch/issues/127176)
and a similar report in Unsloth's issue tracker) caused by a transient
bfloat16 tensor inside bitsandbytes' internal compute path, which crashes
`GradScaler`'s gradient-clipping step. With bf16 unsupported at the hardware
level and fp16 hitting this bug, full fp32 was the only remaining option.
Cost: slower training (no tensor-core-accelerated mixed precision) — offset
here by LoRA's tiny trainable footprint (~0.57% of total parameters) and a
reduced-scope training set sized to fit within free-tier session limits.

**`gradient_checkpointing_kwargs={"use_reentrant": True}`:**
The modern, generally-recommended non-reentrant checkpointing mode
(`use_reentrant=False`) threw a `CheckpointError`
("A different number of tensors was saved during the original forward and
recomputation") — caused by an RNG-state mismatch between the original
forward pass and its recomputation during backward, specifically triggered
by LoRA's dropout combined with bitsandbytes' custom 4-bit autograd
operations. The older reentrant implementation handles RNG-state
preservation differently, in a way compatible with this combination, and
resolved it cleanly.


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
import time

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
HUB_MODEL_ID = "DLCS/contract-clause-phi3-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    attn_implementation="sdpa",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

dataset = load_dataset("json", data_files={
    "train": "data/processed/train.jsonl",
    "validation": "data/processed/val.jsonl",
    "test": "data/processed/test.jsonl",
})

training_args = SFTConfig(
    max_length=1024,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=11,
    num_train_epochs=2,

    eval_strategy="steps",
    eval_steps=25,
    logging_steps=25,
    save_strategy="steps",
    save_steps=25,
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
    completion_only_loss=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",

    output_dir="./outputs/phi3-clause-lora",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_private_repo=True,
    hub_strategy="checkpoint",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)

start = time.time()
trainer.train()
elapsed = time.time() - start

print(f"Total training time: {elapsed/3600:.2f} hours")
trainer.push_to_hub()

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

trainable params: 25,165,824 || all params: 3,846,245,376 || trainable%: 0.6543


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
25,1.299520,0.843730,0.651354,52032.000000,0.791299
50,0.299749,0.650607,0.477825,105941.000000,0.818043
75,0.216908,0.416511,0.550181,159575.000000,0.869679
100,0.207987,0.502180,0.447320,211705.000000,0.854835
125,0.173507,0.525325,0.396716,264348.000000,0.845518
150,0.166218,0.427802,0.418314,317494.000000,0.862024
175,0.170094,0.331467,0.384176,372951.000000,0.893480
200,0.101114,0.617535,0.307320,423960.000000,0.858176
225,0.088929,0.308269,0.252909,478441.000000,0.901618
250,0.078904,0.383511,0.259126,532346.000000,0.890757


Total training time: 3.07 hours


CommitInfo(commit_url='https://huggingface.co/DLCS/contract-clause-phi3-lora/commit/50cd272ed0fff6836946283dd42b29aa3e5acab7', commit_message='End of training', commit_description='', oid='50cd272ed0fff6836946283dd42b29aa3e5acab7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/DLCS/contract-clause-phi3-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='DLCS/contract-clause-phi3-lora'), pr_revision=None, pr_num=None)